In [ ]:
!pip install scikit-learn
!pip install tensorflow
!pip install torch
!pip install xgboost

In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import vstack
import xgboost as xgb

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_json('C:/Users/007pe/Downloads/tdidf_embeddings.json', lines=True)

In [3]:
df = df.dropna(subset=['Speaker_party_name'])
print("DataFrame after dropping NaN in 'Speaker_party_name':")
print(df)

DataFrame after dropping NaN in 'Speaker_party_name':
                Speaker_party_name  \
0       Centre-right to right-wing   
1            Centre to centre-left   
2       Centre-right to right-wing   
3                      Centre-left   
4       Centre-right to right-wing   
...                            ...   
591683                 Centre-left   
591684  Centre-right to right-wing   
591685                 Centre-left   
591686  Centre-right to right-wing   
591687                 Centre-left   

                                                   tokens  \
0       [government, track, deliver, commitment, intro...   
1       [clear, exit, check, scrap, previous, labour, ...   
2       [indicate, original, answer, track, ensure, ex...   
3       [give, situation, border, calais, home, secret...   
4       [great, deal, work, french, authority, relatio...   
...                                                   ...   
591683  [argument, law, protect, everybody, action, ta...   
5

In [4]:
X = np.vstack(df['embedding'].values)
y = df['Speaker_party_name']

In [5]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)  
num_classes = len(encoder.classes_) 

print(encoder.classes_)
print(num_classes)

['Centre to centre-left' 'Centre-left' 'Centre-left to left-wing'
 'Centre-right to right-wing' 'Cleric' 'Independent' 'Left-wing'
 'Left-wing to far-left' 'Non-partisan' 'Right-wing'
 'Right-wing to far-right']
11


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [7]:
xgb_classifier = xgb.XGBClassifier(objective='multi:softmax', num_class=num_classes, random_state=0)

In [8]:
xgb_classifier.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

[0]	validation_0-mlogloss:1.72653
[1]	validation_0-mlogloss:1.49411
[2]	validation_0-mlogloss:1.34553
[3]	validation_0-mlogloss:1.24392
[4]	validation_0-mlogloss:1.17092
[5]	validation_0-mlogloss:1.11604
[6]	validation_0-mlogloss:1.07423
[7]	validation_0-mlogloss:1.04139
[8]	validation_0-mlogloss:1.01550
[9]	validation_0-mlogloss:0.99482
[10]	validation_0-mlogloss:0.97769
[11]	validation_0-mlogloss:0.96377
[12]	validation_0-mlogloss:0.95204
[13]	validation_0-mlogloss:0.94216
[14]	validation_0-mlogloss:0.93352
[15]	validation_0-mlogloss:0.92604
[16]	validation_0-mlogloss:0.91979
[17]	validation_0-mlogloss:0.91416
[18]	validation_0-mlogloss:0.90952
[19]	validation_0-mlogloss:0.90534
[20]	validation_0-mlogloss:0.90166
[21]	validation_0-mlogloss:0.89812
[22]	validation_0-mlogloss:0.89498
[23]	validation_0-mlogloss:0.89203
[24]	validation_0-mlogloss:0.88941
[25]	validation_0-mlogloss:0.88701
[26]	validation_0-mlogloss:0.88472
[27]	validation_0-mlogloss:0.88261
[28]	validation_0-mlogloss:0.8

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None, num_class=11,
              num_parallel_tree=None, ...)

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(xgb_classifier, X_train, y_train, cv=cv, scoring='accuracy')

print("Cross-validation results:")
print(f"Mean Accuracy: {np.mean(scores):.4f} \u00b1 {np.std(scores):.4f}")

print("\nMaking predictions...")
predictions = xgb_classifier.predict(X_test)
proba = xgb_classifier.predict_proba(X_test)

print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Cross-validation results:
Mean Accuracy: 0.6955 ± 0.0016

Making predictions...

Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.12      0.19      6893
           1       0.55      0.52      0.53     28134
           2       0.67      0.29      0.41      6123
           3       0.75      0.91      0.82     69764
           4       0.67      0.21      0.33       456
           5       0.64      0.14      0.23       435
           6       0.73      0.13      0.23       515
           7       0.00      0.00      0.00         3
           8       0.43      0.19      0.26      4155
           9       0.64      0.28      0.39      1607
          10       0.88      0.09      0.16        81

    accuracy                           0.70    118166
   macro avg       0.58      0.26      0.32    118166
weighted avg       0.67      0.70      0.66    118166

Accuracy: 0.6961

Confusion Matrix:
[[  821  2729   101  2951     4     3     7    